In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Load base MoE model
model = AutoModelForCausalLM.from_pretrained("dice-research/lola_v1", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("dice-research/lola_v1",trust_remote_code=True)

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['c_attn', 'c_proj']  # Adjusted to match model's attention modules
)

# Apply LoRA to the model
model = get_peft_model(model, peft_config)

In [ ]:
#model

In [ ]:
# # Load a dataset for training
# dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# # Tokenize the dataset
# def tokenize_function(examples):
#     return tokenizer(examples['text'])

# tokenized_datasets = dataset.map(
#     tokenize_function, 
#     batched=True, 
#     remove_columns=["text"],
#     desc="Tokenizing dataset",
# )

# # Debugging: Check the length of the tokenized dataset
# print(f"Number of training examples after tokenization: {len(tokenized_datasets['train'])}")
# print(f"First tokenized example: {tokenized_datasets['train'][0]}")

# # Adjust block size
# block_size = 64  # Smaller block size for testing purposes

# # Adjust group_texts function
# def group_texts(examples):
#     # Concatenate all texts together
#     concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
#     total_length = len(concatenated_examples['input_ids'])
#     # If total_length is less than block_size, skip this batch
#     if total_length < block_size:
#         return {}
#     # We drop the last chunk if it's smaller than block_size
#     total_length = (total_length // block_size) * block_size
#     # Split the concatenated texts into blocks of block_size
#     result = {
#         k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
#         for k, t in concatenated_examples.items()
#     }
#     return result

# lm_datasets = tokenized_datasets.map(
#     group_texts,
#     batched=True,
#     batch_size=1000,  # Process larger batches to get enough data
#     desc="Grouping texts into blocks",
# )

# # Debugging: Check the length of the dataset after grouping
# print(f"Number of training examples after grouping: {len(lm_datasets['train'])}")
# if len(lm_datasets['train']) > 0:
#     print(f"First grouped example: {lm_datasets['train'][0]}")
# else:
#     print("No training examples after grouping. Consider reducing the block_size or using a larger dataset.")

# # Data collator for language modeling
# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer, 
#     mlm=False
# )

In [ ]:
dataset = load_dataset("json", data_files="alpaca_data.json")
dataset

In [ ]:
# Load the Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca")

# Prepare the prompt template
def generate_prompt(instruction, input_text, response):
    if input_text.strip() != "":
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nResponse:"
    else:
        prompt = f"Instruction: {instruction}\nResponse:"
    full_text = prompt + response + tokenizer.eos_token
    return full_text

# Apply the prompt template to the dataset
def preprocess_function(examples):
    # Extract fields from the examples dict
    instructions = examples['instruction']
    inputs = examples['input']
    outputs = examples['output']
    
    texts = []
    for instruction, input_text, response in zip(instructions, inputs, outputs):
        full_text = generate_prompt(instruction, input_text, response)
        texts.append(full_text)
    
    tokenized_inputs = tokenizer(
        texts,
        max_length=512,  # Adjust max_length as needed
        truncation=True,
        padding='max_length',
    )
    # Create labels by copying the input_ids
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Tokenize the dataset
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing and formatting the dataset",
)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [ ]:
# # Define training arguments
# training_args = TrainingArguments(
#     output_dir='./results',
#     overwrite_output_dir=True,
#     num_train_epochs=3,
#     per_device_train_batch_size=2,
#     save_steps=10_000,
#     save_total_limit=2,
#     logging_steps=500,
#     logging_dir='./logs',
#     learning_rate=5e-5,  # You can adjust the learning rate
#     weight_decay=0.01,   # And weight decay if needed
# )

# # Initialize the Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     data_collator=data_collator,
#     train_dataset=lm_datasets['train'],
#     eval_dataset=lm_datasets['validation']
# )

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,  # Adjust based on your GPU memory
    save_steps=1000,
    save_total_limit=2,
    logging_steps=100,
    logging_dir='./logs',
    learning_rate=5e-5,  # You can adjust the learning rate
    weight_decay=0.01,   # And weight decay if needed
    fp16=torch.cuda.is_available(),  # Enable mixed-precision training if supported
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset['train'],
)

In [ ]:
# Start training
trainer.train()

In [ ]:
model.save_pretrained('./alpaca-peft-model')
tokenizer.save_pretrained('./alpaca-peft-model')